# hexaload — feature exploration

State-of-the-art feature-engineering checks on the processed hourly series (offline, plotly only).

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import hexaload.viz  # noqa: F401 — applies the hexaload plotly template
from hexaload.features import build_feature_table

pio.renderers.default = "notebook"  # plotly.js inline: renders offline

eco = pd.read_parquet("../data/processed/eco2mix-national-hourly.parquet")
wx = pd.read_parquet("../data/processed/openmeteo-icon_eu-hourly-national.parquet")
wx = wx.loc[
    wx.index >= wx["temperature_2m_c"].first_valid_index()
]  # drop NaN head artefact
df = wx.join(eco["consommation"])
df["consommation"] = df["consommation"].interpolate(limit_direction="both")
df = df.dropna(subset=["consommation"]).sort_index()
df = df.join(build_feature_table(df.index)[0])
loc = df.index.tz_convert("Europe/Paris")
print(f"{df.index.min()} -> {df.index.max()} | {len(df):,} h | {df.shape[1]} cols")

## 1. Seasonality

In [ ]:
prof = df["consommation"].groupby([loc.day_of_week, loc.hour]).mean().unstack()
prof.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
px.imshow(
    prof,
    labels=dict(x="local hour", y="weekday", color="MW"),
    title="Average hourly load profile by weekday",
    aspect="auto",
)

In [ ]:
prof_m = df["consommation"].groupby([loc.month, loc.hour]).mean().unstack()
px.imshow(
    prof_m,
    labels=dict(x="local hour", y="month", color="MW"),
    title="Average hourly load profile by month",
    aspect="auto",
)

In [ ]:
cols = {}
for d, name in [
    ("2024-03-31", "summer"),
    ("2024-10-27", "winter"),
    ("2025-03-30", "summer"),
    ("2025-10-26", "winter"),
    ("2026-03-29", "summer"),
    ("2026-10-25", "winter"),
]:
    t0 = pd.Timestamp(d, tz="UTC")
    sub = df["consommation"].loc[
        t0 - pd.Timedelta(hours=48) : t0 + pd.Timedelta(hours=47)
    ]
    cols[f"{d} ({name})"] = pd.Series(sub.to_numpy(), index=range(-48, 48))
px.line(
    pd.DataFrame(cols),
    title="Hourly load around DST switches",
    labels={"x": "hours relative to switch (UTC)", "y": "MW"},
)

## 2. Calendar

In [ ]:
typ = pd.Series("workday", index=df.index, name="day_type")
typ[df["cal_pont"] == 1] = "bridge"
vac = (
    (df["cal_vacances_zone_a"] == 1)
    | (df["cal_vacances_zone_b"] == 1)
    | (df["cal_vacances_zone_c"] == 1)
)
typ[vac] = "school vacation"
typ[df["cal_ferie"] == 1] = "holiday"
px.box(
    df[["consommation"]].join(typ),
    x="day_type",
    y="consommation",
    category_orders={"day_type": ["workday", "bridge", "school vacation", "holiday"]},
    title="Load distribution by day type",
)

In [ ]:
flags = [
    "cal_ferie",
    "cal_veille_ferie",
    "cal_lendemain_ferie",
    "cal_pont",
    "cal_vacances_zone_a",
    "cal_vacances_zone_b",
    "cal_vacances_zone_c",
    "cal_heure_ete",
]
moy = df["consommation"].mean()
ec = pd.DataFrame(
    [
        {
            "flag": c,
            "rel_change_%": 100
            * (df.loc[df[c] == 1, "consommation"].mean() - moy)
            / moy,
        }
        for c in flags
    ]
).sort_values("rel_change_%")
px.bar(
    ec,
    x="rel_change_%",
    y="flag",
    orientation="h",
    title="Average relative load change when the flag is active (%)",
    labels={"rel_change_%": "relative change (%)"},
)

## 3. Weather (archived ICON-EU forecasts, as-of)

In [ ]:
T = df["temperature_2m_c"]
bins = pd.cut(T, bins=np.arange(-10, 42, 2))
u = df.groupby(bins, observed=True)["consommation"].mean()
u.index = [iv.mid for iv in u.index]
fig = px.line(
    u,
    title="Average load by 2m temperature (U-shape: heating/cooling)",
    labels={"x": "2m temperature (°C)", "y": "MW"},
)
fig.show()
samp = df.sample(min(20_000, len(df)), random_state=0)
px.scatter(
    samp,
    x="temperature_2m_c",
    y="consommation",
    opacity=0.15,
    title="Load vs temperature scatter (20k sample)",
    labels={"temperature_2m_c": "temperature (°C)", "consommation": "MW"},
)

In [ ]:
y = df["consommation"]
cc = pd.Series(
    {
        k: (
            np.corrcoef(y.iloc[k:], T.iloc[: len(T) - k])[0, 1]
            if k
            else np.corrcoef(y, T)[0, 1]
        )
        for k in range(0, 73)
    }
)
px.line(
    cc,
    title="Correlation load(t) vs temperature(t-k) — thermal inertia",
    labels={"x": "lag k (h)", "y": "Pearson correlation"},
)

In [ ]:
wxcols = [
    c
    for c in df.columns
    if c.split("_")[-1] in ("c", "pct", "hpa", "mm", "ms", "wm2", "fraction", "code")
]
rows = [
    {
        "var": c,
        "pearson": df[c].corr(df["consommation"]),
        "spearman": df[c].corr(df["consommation"], method="spearman"),
    }
    for c in wxcols
]
px.bar(
    pd.DataFrame(rows).melt(id_vars="var"),
    x="value",
    y="var",
    color="variable",
    barmode="group",
    orientation="h",
    title="Raw weather variables vs load: correlation",
    labels={"value": "correlation", "var": ""},
)

## 4. Lags & load memory

In [ ]:
def acf_series(s: pd.Series, max_lag: int) -> pd.Series:
    v = s.to_numpy(dtype=float) - s.mean()
    denom = float(np.dot(v, v))
    return pd.Series(
        {
            k: (float(np.dot(v[:-k], v[k:]) / denom) if k else 1.0)
            for k in range(max_lag + 1)
        }
    )


a = acf_series(df["consommation"].loc["2023":], 336)
fig = px.line(a, title="Load ACF, lags 0..336 h", labels={"x": "lag (h)", "y": "ACF"})
fig.add_vline(x=24, line_dash="dash", annotation_text="24h")
fig.add_vline(x=168, line_dash="dash", annotation_text="168h")
fig

In [ ]:
lag168 = df["consommation"].shift(168)
samp = df.sample(20_000, random_state=0)
px.scatter(
    x=lag168.loc[samp.index],
    y=samp["consommation"],
    color=samp.index.tz_convert("Europe/Paris").hour,
    opacity=0.25,
    title="load(t) vs load(t-168h), colored by local hour",
    labels={"x": "load(t-168h) MW", "y": "load(t) MW", "color": "hour"},
)

## 5. Permutation importance (hand-rolled linear ridge, 2026 holdout)

In [ ]:
T = df["temperature_2m_c"]
derived = pd.DataFrame(
    {
        "hdd16": (16.0 - T).clip(lower=0),
        "cdd24": (T - 24.0).clip(lower=0),
        "t_min_day": T.groupby(loc.normalize()).transform("min"),
        "t_max_day": T.groupby(loc.normalize()).transform("max"),
        "lag24": df["consommation"].shift(24),
        "lag168": df["consommation"].shift(168),
    },
    index=df.index,
)
Xf = df[list(build_feature_table(df.index)[0].columns)].join(derived).dropna()
y = df["consommation"].reindex(Xf.index)
train = Xf.index < "2026-01-01"
mu, sd = Xf.loc[train].mean(), Xf.loc[train].std().replace(0, 1.0)
Z = ((Xf - mu) / sd).to_numpy()
yv = y.to_numpy()
Zt = np.c_[np.ones(train.sum()), Z[train]]
theta, *_ = np.linalg.lstsq(Zt, yv[train], rcond=None)
Ztst = np.c_[np.ones((~train).sum()), Z[~train]]
base = float(np.mean(np.abs(yv[~train] - Ztst @ theta)))
rng = np.random.default_rng(0)
cols = list(Xf.columns)
imp = {}
for j, c in enumerate(cols):
    d = []
    for _ in range(10):
        Xp = Ztst.copy()
        Xp[:, 1 + j] = Ztst[rng.permutation(len(Ztst)), 1 + j]
        d.append(float(np.mean(np.abs(yv[~train] - Xp @ theta))) - base)
    imp[c] = float(np.mean(d))
px.bar(
    pd.Series(imp).sort_values(),
    orientation="h",
    title=f"Permutation importance — ridge, base MAE 2026 = {base:.0f} MW",
    labels={"x": "delta MAE (MW)", "y": ""},
)

## 6. Feature multicollinearity (pre-mRMR / SHAP-in-the-loop)

In [ ]:
corr = Xf.corr().round(2)
px.imshow(
    corr,
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    aspect="auto",
    title="Feature correlation matrix",
)